# Integration experiment and ablation (GPU)
#
Does the content signal improve propagation-based detection? The content
sub-module produces a per-story credibility score from the news text
embedding; the score is injected as one extra feature on the news root node
of the propagation graph, and Bi-GCN is retrained.
#
Three settings per configuration: **graph only**, **+ random score**
(control: must change nothing), **+ content score**. Two node-feature sets
(`profile`, no content anywhere; `bert`, content already in the root
embedding) on both UPFD datasets, three seeds each.
#
Content scores are **cross-fitted** (5-fold on the training split) so that no
training graph ever receives a score predicted by a model that saw its label.

In [ ]:
%pip install -q torch-geometric

import json

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.nn import Linear
from torch_geometric.data import Data
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

assert torch.cuda.is_available(), "No GPU: pick a T4 runtime"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
ROOT = "/tmp/upfd"
SEEDS = (0, 1, 2)

## Models

In [ ]:
class BiGCN(torch.nn.Module):
    """Bidirectional GCN (Bian et al., 2020): the cascade read top-down along the
    propagation edges and bottom-up along the reversed edges."""

    def __init__(self, in_dim, hidden, num_classes):
        super().__init__()
        self.td1, self.td2 = GCNConv(in_dim, hidden), GCNConv(hidden, hidden)
        self.bu1, self.bu2 = GCNConv(in_dim, hidden), GCNConv(hidden, hidden)
        self.lin = Linear(hidden * 2, num_classes)

    def forward(self, x, edge_index, batch):
        rev = edge_index.flip(0)
        td = F.relu(self.td2(F.relu(self.td1(x, edge_index)), edge_index))
        bu = F.relu(self.bu2(F.relu(self.bu1(x, rev)), rev))
        return self.lin(torch.cat([global_mean_pool(td, batch),
                                   global_mean_pool(bu, batch)], dim=1))


class ContentMLP(torch.nn.Module):
    """Content sub-module: reads the news root embedding only (no structure)."""

    def __init__(self, in_dim, hidden=128):
        super().__init__()
        self.l1, self.l2 = Linear(in_dim, hidden), Linear(hidden, 2)

    def forward(self, x):
        return self.l2(F.dropout(F.relu(self.l1(x)), p=0.3, training=self.training))

## Content sub-module: train, evaluate, and cross-fit the scores

In [ ]:
def roots_and_labels(dataset):
    """Root-node embedding (the news content) and label of every graph."""
    x = torch.stack([d.x[0] for d in dataset])
    y = torch.tensor([int(d.y) for d in dataset])
    return x, y


def fit_content(x_tr, y_tr, epochs=120, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    model = ContentMLP(x_tr.size(1)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    x_tr, y_tr = x_tr.to(DEVICE), y_tr.to(DEVICE)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        F.cross_entropy(model(x_tr), y_tr).backward()
        opt.step()
    return model


@torch.no_grad()
def credibility(model, x):
    """P(reliable) in [0, 1], the score handed to the graph sub-module."""
    model.eval()
    return F.softmax(model(x.to(DEVICE)), dim=1)[:, 1].cpu()


def content_scores(train_ds, test_ds, seed=0):
    """Cross-fitted scores on train (5-fold), direct scores on test."""
    x_tr, y_tr = roots_and_labels(train_ds)
    x_te, y_te = roots_and_labels(test_ds)

    oof = torch.zeros(len(y_tr))
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for fit_idx, held_idx in folds.split(x_tr.numpy(), y_tr.numpy()):
        m = fit_content(x_tr[fit_idx], y_tr[fit_idx], seed=seed)
        oof[held_idx] = credibility(m, x_tr[held_idx])

    full = fit_content(x_tr, y_tr, seed=seed)
    te = credibility(full, x_te)
    pred = (te > 0.5).long()
    metrics = {"accuracy": round(accuracy_score(y_te, pred), 3),
               "macro_f1": round(f1_score(y_te, pred, average="macro"), 3)}
    return oof, te, metrics

## Graph sub-module with the injected score

In [ ]:
def attach(dataset, scores):
    out = []
    for data, s in zip(dataset, scores, strict=True):
        col = torch.zeros(data.num_nodes, 1)
        col[0, 0] = float(s)  # the news root carries the credibility score
        out.append(Data(x=torch.cat([data.x, col], dim=1),
                        edge_index=data.edge_index, y=data.y))
    return out


def run_graph(train_list, test_list, in_dim, epochs=60, hidden=64, lr=0.01, seed=0):
    torch.manual_seed(seed)
    model = BiGCN(in_dim, hidden, 2).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    tr = DataLoader(train_list, batch_size=128, shuffle=True)
    te = DataLoader(test_list, batch_size=128)
    for _ in range(epochs):
        model.train()
        for b in tr:
            b = b.to(DEVICE)
            opt.zero_grad()
            F.cross_entropy(model(b.x, b.edge_index, b.batch), b.y).backward()
            opt.step()
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for b in te:
            b = b.to(DEVICE)
            ps += model(b.x, b.edge_index, b.batch).argmax(dim=1).cpu().tolist()
            ys += b.y.cpu().tolist()
    return accuracy_score(ys, ps), f1_score(ys, ps, average="macro")

## Experiment

In [ ]:
results, content_summary = [], []

for name in ("politifact", "gossipcop"):
    # the content sub-module always reads the text embedding of the news
    tr_bert = UPFD(ROOT, name, "bert", split="train")
    te_bert = UPFD(ROOT, name, "bert", split="test")
    print(f"\n===== {name}  (train {len(tr_bert)}, test {len(te_bert)}) =====")

    per_seed_scores = {}
    for seed in SEEDS:
        oof, te_s, m = content_scores(tr_bert, te_bert, seed=seed)
        per_seed_scores[seed] = (oof, te_s)
        content_summary.append({"dataset": name, "seed": seed, "model": "content only", **m})
        print(f"  content-only (seed {seed}): acc {m['accuracy']:.3f}  "
              f"macro-F1 {m['macro_f1']:.3f}")

    for feature in ("profile", "bert"):
        tr = tr_bert if feature == "bert" else UPFD(ROOT, name, feature, split="train")
        te = te_bert if feature == "bert" else UPFD(ROOT, name, feature, split="test")
        in_dim = tr.num_features + 1
        for setting in ("graph only", "+ random score", "+ content score"):
            accs, f1s = [], []
            for seed in SEEDS:
                oof, te_s = per_seed_scores[seed]
                g = torch.Generator().manual_seed(seed)
                if setting == "graph only":
                    s_tr, s_te = torch.zeros(len(tr)), torch.zeros(len(te))
                elif setting == "+ random score":
                    s_tr, s_te = torch.rand(len(tr), generator=g), torch.rand(len(te), generator=g)
                else:
                    s_tr, s_te = oof, te_s
                a, f = run_graph(attach(tr, s_tr), attach(te, s_te), in_dim, seed=seed)
                accs.append(a)
                f1s.append(f)
            row = {"dataset": name, "features": feature, "setting": setting,
                   "accuracy": round(float(np.mean(accs)), 3),
                   "macro_f1": round(float(np.mean(f1s)), 3),
                   "macro_f1_std": round(float(np.std(f1s)), 3)}
            results.append(row)
            print(f"  {feature:8s} {setting:16s} macro-F1 {row['macro_f1']:.3f}"
                  f" +/- {row['macro_f1_std']:.3f}")

## Summary

In [ ]:
print(f"\n{'dataset':11s} {'features':9s} {'setting':16s} {'macro-F1':>9s} {'std':>6s}")
print("-" * 56)
for r in results:
    print(f"{r['dataset']:11s} {r['features']:9s} {r['setting']:16s} "
          f"{r['macro_f1']:>9.3f} {r['macro_f1_std']:>6.3f}")

print("\ncontent-only (mean over seeds)")
for name in ("politifact", "gossipcop"):
    rows = [c for c in content_summary if c["dataset"] == name]
    print(f"  {name:11s} macro-F1 {np.mean([r['macro_f1'] for r in rows]):.3f}"
          f"  acc {np.mean([r['accuracy'] for r in rows]):.3f}")

with open("integration_metrics.json", "w") as f:
    summary = {"seeds": list(SEEDS), "graph": results, "content_only": content_summary}
    json.dump(summary, f, indent=2)
print("\nsaved integration_metrics.json")